# 龟龟港股选股器 (Turtle HK Screener)

## 两层架构批量筛选器 — 港股版

从全港股 ~2500 只中，用策略所有可量化指标快速过滤候选标的。

### 流水线总览

```
全港股 ~2500只
    │
    ▼
┌─────────────────────────────────┐
│  Tier 1: 批量粗筛              │
│  Phase A: hk_basic + hk_daily  │
│    → 上市年限、成交额预筛      │
│  Phase B: hk_fina_indicator    │
│    → PE/PB/市值/股息率         │
│  Phase C: 双通道排序           │
│    → 主通道100 + 观察通道30    │
└─────────────────────────────────┘
    │  ~130只
    ▼
┌─────────────────────────────────┐
│  Tier 2: 逐股深度分析          │
│  • 硬性否决: 负债率>85%        │
│  • 质量门槛: ROE/毛利率/负债率 │
│  • Factor 2: 穿透回报率 R      │
│  • Factor 4: EV/EBITDA, FCF    │
│  • 底价: 5种方法               │
└─────────────────────────────────┘
    │
    ▼
  综合评分 Top N
```

### 与A股版差异

| 维度 | A股版 | 港股版 |
|------|-------|--------|
| Tier 1 数据源 | `stock_basic` + `daily_basic` (2次API) | `hk_basic` + `hk_daily` + `hk_fina_indicator` (3阶段) |
| 估值数据 | `daily_basic` 实时 PE/PB/股息率 | `hk_fina_indicator` 报告期数据 |
| 硬性否决 | 质押比例 + 审计意见 | 负债率否决 (港股无质押/审计API) |
| 财报格式 | 结构化字段 | 行转列 (`ind_name`/`ind_value`) 需 pivot |
| 货币 | 人民币 (百万元) | 港币 (百万港元) |
| ST股过滤 | 有 | 无 (港股无ST概念) |
| 行业分类 | `stock_basic.industry` | 无 (hk_basic无行业字段) |

## Step 1: 环境初始化与参数配置

下方代码完成两件事：**加载依赖** 和 **设定筛选参数**。

所有参数均可自定义修改。默认值及其含义：

### Tier 1 硬性过滤

| 参数 | 默认值 | 含义 |
|------|--------|------|
| `min_listing_years` | 3 | 排除次新股，确保至少3年完整年报数据可供分析 |
| `min_market_cap_hkm` | 5000.0 | 最低市值（百万港元，即50亿港元），排除微盘股 |
| `min_daily_amount_hkm` | 5.0 | 最低日成交额（百万港元），排除流动性极差的股票 |
| `max_pb` | 10.0 | 市净率上限，排除泡沫股 |
| `max_pe` | 50.0 | 市盈率上限（主通道），排除高估值投机标的 |

### Tier 1 双通道

| 参数 | 默认值 | 含义 |
|------|--------|------|
| `obs_channel_limit` | 30 | **观察通道**：`PE < 0`（当前亏损但可能有反转价值，按市值取前30只） |
| `tier2_main_limit` | 100 | **主通道** 入 Tier 2 最大数量 |

### Tier 1 排序权重

| 参数 | 默认值 | 含义 |
|------|--------|------|
| `dv_weight` | 0.4 | 股息率权重（越高越好） |
| `pe_weight` | 0.3 | 1/PE 权重（越低 PE 越好） |
| `pb_weight` | 0.3 | 1/PB 权重（越低 PB 越好） |

### Tier 2 硬性否决

| 参数 | 默认值 | 含义 |
|------|--------|------|
| `max_debt_ratio_veto` | 85.0 | 资产负债率否决线(%)，超过一票否决（替代A股的质押+审计否决） |

### Tier 2 质量门槛

| 参数 | 默认值 | 含义 |
|------|--------|------|
| `min_roe` | 8.0 | ROE 最低门槛 (%) |
| `min_gross_margin` | 15.0 | 毛利率下限 (%) |
| `max_debt_ratio` | 70.0 | 资产负债率上限 (%) |

## Core Engine

下方代码定义了港股选股器的完整引擎，包含四个核心组件：

| 组件 | 作用 |
|------|------|
| `HKScreenerConfig` | 所有可调参数的数据类（阈值、权重、缓存设置） |
| `ScreenerCache` | Parquet 磁盘缓存，按 TTL 自动失效 |
| `HK_*_MAP` + `_pivot_hk_line_items()` | 港股行转列财报格式转换 |
| `HKScreener` | 筛选器主类：Tier 1 批量筛选 + Tier 2 逐股深度分析 |

In [ ]:
# Engine: HKScreenerConfig + ScreenerCache + HKScreener (self-contained)
# ---------------------------------------------------------------
# This cell defines the full HK screening engine so the notebook is
# self-contained — no imports from scripts/ are needed.
# ---------------------------------------------------------------

from __future__ import annotations

import hashlib
import os
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta
from typing import Any

import pandas as pd


# ============================================================
# Project root detection (robust across cwd variations)
# ============================================================

def _find_project_root() -> str:
    """Walk up from cwd to find project root (contains .git/)."""
    d = os.path.abspath('')
    for _ in range(5):
        if os.path.isdir(os.path.join(d, '.git')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath('')

_PROJECT_ROOT = _find_project_root()


# ============================================================
# .env loader & token helper (from scripts/config.py)
# ============================================================

def _load_env_file() -> None:
    """Load .env file from project root if it exists."""
    env_path = os.path.join(_PROJECT_ROOT, '.env')
    if not os.path.isfile(env_path):
        return
    with open(env_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                key, _, value = line.partition('=')
                key = key.strip()
                value = value.strip().strip('"\'')
                if key and key not in os.environ:
                    os.environ[key] = value

_load_env_file()


def _get_token() -> str:
    token = os.environ.get('TUSHARE_TOKEN', '')
    if not token:
        raise RuntimeError('TUSHARE_TOKEN not set. Run: export TUSHARE_TOKEN=your_token')
    return token


# ============================================================
# HK Line-Item Field Mappings (from tushare_modules/constants.py)
# ============================================================

HK_INCOME_MAP = {
    'revenue': '营业额',
    'oper_cost': '营运支出',
    'sell_exp': '销售及分销费用',
    'admin_exp': '行政开支',
    'operate_profit': '经营溢利',
    'invest_income': '应占联营公司溢利',
    'int_income': '利息收入',
    'finance_exp': '融资成本',
    'total_profit': '除税前溢利',
    'income_tax': '税项',
    'n_income': '除税后溢利',
    'n_income_attr_p': '股东应占溢利',
    'minority_gain': '少数股东损益',
    'basic_eps': '每股基本盈利',
    'diluted_eps': '每股摊薄盈利',
}

HK_BALANCE_MAP = {
    'money_cap': '现金及等价物',
    'accounts_receiv': '应收帐款',
    'inventories': '存货',
    'total_cur_assets': '流动资产合计',
    'lt_eqt_invest': '联营公司权益',
    'fix_assets': '物业厂房及设备',
    'cip': '在建工程',
    'intang_assets': '无形资产',
    'total_assets': '总资产',
    'acct_payable': '应付帐款',
    'notes_payable': '应付票据',
    'contract_liab': '递延收入(流动)',
    'st_borr': '短期贷款',
    'total_cur_liab': '流动负债合计',
    'lt_borr': '长期贷款',
    'bond_payable': '应付票据(非流动)',
    'total_liab': '总负债',
    'defer_tax_assets': '递延税项资产',
    'defer_tax_liab': '递延税项负债',
    'total_hldr_eqy_exc_min_int': '股东权益',
    'minority_int': '少数股东权益',
}

HK_CASHFLOW_MAP = {
    'n_cashflow_act': '经营业务现金净额',
    'n_cashflow_inv_act': '投资业务现金净额',
    'n_cash_flows_fnc_act': '融资业务现金净额',
    'c_pay_acq_const_fiolta': '购建无形资产及其他资产',
    'depr_fa_coga_dpba': '折旧及摊销',
    'c_pay_dist_dpcp_int_exp': '已付股息(融资)',
    'c_paid_for_taxes': '已付税项',
    'c_recp_return_invest': '收回投资所得现金',
}


def _pivot_hk_line_items(df: pd.DataFrame, field_map: dict) -> pd.DataFrame:
    """Pivot HK ind_name/ind_value rows into one-row-per-period columns."""
    if df.empty or 'ind_name' not in df.columns:
        return pd.DataFrame()
    reverse_map = {v: k for k, v in field_map.items() if v is not None}
    df_mapped = df[df['ind_name'].isin(reverse_map)].copy()
    if df_mapped.empty:
        return pd.DataFrame()
    df_mapped['field'] = df_mapped['ind_name'].map(reverse_map)
    df_mapped['ind_value'] = pd.to_numeric(df_mapped['ind_value'], errors='coerce')
    pivoted = df_mapped.pivot_table(
        index=['end_date', 'ts_code'], columns='field',
        values='ind_value', aggfunc='first'
    ).reset_index()
    pivoted.columns.name = None
    return pivoted


# ============================================================
# HKScreenerConfig
# ============================================================

@dataclass
class HKScreenerConfig:
    """All tunable parameters for the HK screener pipeline."""

    # --- Tier 1: Hard filters ---
    min_listing_years: int = 3
    min_market_cap_hkm: float = 5000.0  # 百万港元 (~50亿港元)
    min_daily_amount_hkm: float = 5.0   # 百万港元 日成交额
    max_pb: float = 10.0
    max_pe: float = 50.0

    # --- Tier 1: Dual-channel PE ---
    obs_channel_limit: int = 30

    # --- Tier 1: Ranking & cutoff ---
    tier2_main_limit: int = 100
    dv_weight: float = 0.4
    pe_weight: float = 0.3
    pb_weight: float = 0.3

    # --- Tier 2: Hard vetoes ---
    max_debt_ratio_veto: float = 85.0  # replaces pledge/audit check

    # --- Tier 2: Financial quality ---
    min_roe: float = 8.0             # %
    min_gross_margin: float = 15.0   # %
    max_debt_ratio: float = 70.0     # %

    # --- Scoring weights ---
    weight_roe: float = 0.20
    weight_fcf_yield: float = 0.20
    weight_penetration_r: float = 0.25
    weight_ev_ebitda: float = 0.15
    weight_floor_premium: float = 0.20

    # --- Cache ---
    cache_dir: str = 'output/.screener_hk_cache'
    cache_hk_basic_ttl_days: int = 7
    cache_hk_daily_ttl_days: int = 0   # same-day only
    cache_fina_indicator_ttl_hours: int = 168  # 7 days (quarterly data)
    cache_tier2_financial_ttl_hours: int = 168
    cache_tier2_market_ttl_hours: int = 24
    cache_tier2_global_ttl_hours: int = 24

    @property
    def tier2_max_stocks(self) -> int:
        return self.tier2_main_limit + self.obs_channel_limit

    @property
    def scoring_weights(self) -> dict[str, float]:
        return {
            'roe': self.weight_roe,
            'fcf_yield': self.weight_fcf_yield,
            'penetration_r': self.weight_penetration_r,
            'ev_ebitda': self.weight_ev_ebitda,
            'floor_premium': self.weight_floor_premium,
        }

    def validate(self) -> list[str]:
        errors = []
        w_sum = (self.weight_roe + self.weight_fcf_yield +
                 self.weight_penetration_r + self.weight_ev_ebitda +
                 self.weight_floor_premium)
        if abs(w_sum - 1.0) > 0.01:
            errors.append(f'Scoring weights must sum to 1.0, got {w_sum:.3f}')
        if self.min_listing_years < 0:
            errors.append('min_listing_years must be >= 0')
        if self.min_market_cap_hkm < 0:
            errors.append('min_market_cap_hkm must be >= 0')
        if self.tier2_main_limit < 1:
            errors.append('tier2_main_limit must be >= 1')
        if self.obs_channel_limit < 0:
            errors.append('obs_channel_limit must be >= 0')
        return errors

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> 'HKScreenerConfig':
        valid_keys = {f.name for f in cls.__dataclass_fields__.values()}
        filtered = {k: v for k, v in d.items() if k in valid_keys}
        return cls(**filtered)


# ============================================================
# ScreenerCache (same as A-share version)
# ============================================================

class ScreenerCache:
    """Parquet-based disk cache with TTL."""

    def __init__(self, cache_dir: str):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def _path(self, key: str) -> str:
        safe_key = hashlib.md5(key.encode()).hexdigest()
        return os.path.join(self.cache_dir, f'{safe_key}.parquet')

    def _meta_path(self, key: str) -> str:
        safe_key = hashlib.md5(key.encode()).hexdigest()
        return os.path.join(self.cache_dir, f'{safe_key}.meta')

    def get(self, key: str, ttl_seconds: int) -> pd.DataFrame | None:
        path = self._path(key)
        meta_path = self._meta_path(key)
        if not os.path.exists(path) or not os.path.exists(meta_path):
            return None
        try:
            with open(meta_path) as f:
                ts = float(f.read().strip().split('\n')[0])
            if time.time() - ts > ttl_seconds:
                return None
            return pd.read_parquet(path)
        except Exception:
            return None

    def put(self, key: str, df: pd.DataFrame) -> None:
        path = self._path(key)
        meta_path = self._meta_path(key)
        try:
            df.to_parquet(path, index=False)
            with open(meta_path, 'w') as f:
                f.write(f'{time.time()}\n{key}')
        except Exception:
            pass

    def invalidate(self, key: str) -> None:
        for p in [self._path(key), self._meta_path(key)]:
            if os.path.exists(p):
                os.remove(p)

    def invalidate_prefix(self, prefix: str) -> None:
        if not os.path.isdir(self.cache_dir):
            return
        for f in os.listdir(self.cache_dir):
            if not f.endswith('.meta'):
                continue
            fp = os.path.join(self.cache_dir, f)
            try:
                with open(fp) as fh:
                    lines = fh.read().strip().split('\n')
                original_key = lines[1] if len(lines) > 1 else ''
                if original_key.startswith(prefix):
                    os.remove(fp)
                    parquet = fp.replace('.meta', '.parquet')
                    if os.path.exists(parquet):
                        os.remove(parquet)
            except Exception:
                pass

    def clear(self) -> None:
        if os.path.isdir(self.cache_dir):
            for f in os.listdir(self.cache_dir):
                fp = os.path.join(self.cache_dir, f)
                if os.path.isfile(fp):
                    os.remove(fp)


# ============================================================
# Tier 2 field supersets & TTL categories
# ============================================================

_HK_TIER2_FIELDS = {
    'hk_income': 'ts_code,end_date,ind_name,ind_value',
    'hk_balancesheet': 'ts_code,end_date,ind_name,ind_value',
    'hk_cashflow': 'ts_code,end_date,ind_name,ind_value',
    'hk_fina_indicator': ('ts_code,end_date,roe_avg,gross_profit_ratio,'
                          'net_profit_ratio,debt_asset_ratio,'
                          'pe_ttm,pb_ttm,total_market_cap,hksk_market_cap,'
                          'operate_income_yoy,holder_profit_yoy,'
                          'bps,basic_eps,diluted_eps,dps_hkd,divi_ratio,dividend_rate'),
    'yc_cb': 'trade_date,yield',
}

_HK_TIER2_TTL_CATEGORY = {
    'hk_income': 'financial',
    'hk_balancesheet': 'financial',
    'hk_cashflow': 'financial',
    'hk_fina_indicator': 'financial',
    'yc_cb': 'global',
}


# ============================================================
# HKScreener
# ============================================================

class HKScreener:
    """Main HK screener: Tier 1 bulk screening + Tier 2 deep analysis."""

    def __init__(self, token: str | None = None, config: HKScreenerConfig | None = None):
        self.config = config or HKScreenerConfig()
        self._token = token or _get_token()
        self._pro = None
        self.cache = ScreenerCache(self.config.cache_dir)
        self._rf_cache: float | None = None
        self._stock_data_cache: dict[str, pd.DataFrame] = {}
        self._ranked_df: pd.DataFrame | None = None

    def _get_pro(self):
        if self._pro is None:
            import tushare as ts
            ts.set_token(self._token)
            self._pro = ts.pro_api(timeout=30)
            api_url = os.environ.get('TUSHARE_API_URL', '')
            if api_url:
                self._pro._DataApi__http_url = api_url
        return self._pro

    def _safe_call(self, api_name: str, **kwargs) -> pd.DataFrame:
        pro = self._get_pro()
        last_err = None
        for attempt in range(1, 4):
            try:
                time.sleep(0.3)
                api_func = getattr(pro, api_name)
                return api_func(**kwargs)
            except Exception as e:
                last_err = e
                if attempt < 3:
                    import tushare as ts
                    self._pro = ts.pro_api(timeout=30)
                    api_url = os.environ.get('TUSHARE_API_URL', '')
                    if api_url:
                        self._pro._DataApi__http_url = api_url
                    time.sleep(1.0 * attempt)
        raise RuntimeError(
            f'Tushare API {api_name!r} failed after 3 retries: {last_err}')

    def _cached_call(self, api_name: str, ts_code: str | None = None,
                     **kwargs) -> pd.DataFrame:
        """Wrapper with in-memory + disk caching."""
        if ts_code is not None:
            cache_key = f'hk_tier2_{ts_code}_{api_name}'
        else:
            cache_key = f'hk_global_{api_name}'

        if cache_key in self._stock_data_cache:
            return self._stock_data_cache[cache_key]

        category = _HK_TIER2_TTL_CATEGORY.get(api_name, 'financial')
        cfg = self.config
        if category == 'financial':
            ttl_seconds = cfg.cache_tier2_financial_ttl_hours * 3600
        elif category == 'market':
            ttl_seconds = cfg.cache_tier2_market_ttl_hours * 3600
        else:
            ttl_seconds = cfg.cache_tier2_global_ttl_hours * 3600

        disk_df = self.cache.get(cache_key, ttl_seconds)
        if disk_df is not None:
            self._stock_data_cache[cache_key] = disk_df
            return disk_df

        call_kwargs = dict(kwargs)
        if api_name in _HK_TIER2_FIELDS:
            call_kwargs['fields'] = _HK_TIER2_FIELDS[api_name]
        if ts_code is not None:
            call_kwargs['ts_code'] = ts_code

        df = self._safe_call(api_name, **call_kwargs)

        if not df.empty:
            self._stock_data_cache[cache_key] = df
            self.cache.put(cache_key, df)

        return df

    def _clear_stock_cache(self, ts_code: str) -> None:
        prefix = f'hk_tier2_{ts_code}_'
        keys_to_remove = [k for k in self._stock_data_cache if k.startswith(prefix)]
        for k in keys_to_remove:
            del self._stock_data_cache[k]

    # ---- Tier 1: Trade date ----

    def _get_latest_hk_trade_date(self) -> str:
        """Get the latest HK trading date."""
        now = datetime.now()
        if now.hour < 19:
            end = (now - timedelta(days=1)).strftime('%Y%m%d')
        else:
            end = now.strftime('%Y%m%d')
        start = (now - timedelta(days=10)).strftime('%Y%m%d')
        try:
            df = self._safe_call('hk_tradecal',
                                 start_date=start, end_date=end,
                                 fields='cal_date,is_open')
            if not df.empty:
                open_days = df[df['is_open'] == 1].sort_values('cal_date', ascending=False)
                if not open_days.empty:
                    return open_days.iloc[0]['cal_date']
        except Exception:
            pass
        return end

    # ---- Tier 1: Bulk data (3-phase) ----

    def _tier1_bulk_data(self, force_refresh: bool = False) -> pd.DataFrame:
        """Phase A: hk_basic + hk_daily merged."""
        cfg = self.config
        trade_date = self._get_latest_hk_trade_date()
        print(f'  最新交易日: {trade_date}')

        # --- hk_basic ---
        sb_key = 'hk_basic_all'
        sb_ttl = cfg.cache_hk_basic_ttl_days * 86400
        stock_df = None if force_refresh else self.cache.get(sb_key, sb_ttl)
        if stock_df is None:
            stock_df = self._safe_call('hk_basic',
                                       fields='ts_code,name,fullname,list_date,market')
            if not stock_df.empty:
                self.cache.put(sb_key, stock_df)

        # --- hk_daily ---
        db_key = f'hk_daily_{trade_date}'
        db_ttl = 18 * 3600 if cfg.cache_hk_daily_ttl_days == 0 else cfg.cache_hk_daily_ttl_days * 86400
        daily_df = None if force_refresh else self.cache.get(db_key, db_ttl)
        if daily_df is None:
            daily_df = self._safe_call('hk_daily', trade_date=trade_date,
                                       fields='ts_code,trade_date,close,vol,amount')
            if not daily_df.empty:
                self.cache.put(db_key, daily_df)

        if stock_df.empty or daily_df.empty:
            return pd.DataFrame()

        merged = stock_df.merge(daily_df, on='ts_code', how='inner')
        return merged

    def _tier1_pre_filter(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply cheap filters before hk_fina_indicator enrichment."""
        if df.empty:
            return df

        cfg = self.config
        today = datetime.now()

        # 1. Remove suspended stocks (vol == 0)
        df = df[df['vol'].notna() & (df['vol'] > 0)].copy()

        # 2. Listing age >= min_listing_years
        cutoff = (today - timedelta(days=cfg.min_listing_years * 365)).strftime('%Y%m%d')
        df = df[df['list_date'].notna()].copy()
        df = df[df['list_date'] <= cutoff].copy()

        # 3. Daily amount >= min_daily_amount_hkm (amount in HKD thousands, convert to millions)
        df = df[df['amount'].notna()].copy()
        df['amount_hkm'] = df['amount'] / 1000  # Tushare hk_daily amount is in thousands HKD
        df = df[df['amount_hkm'] >= cfg.min_daily_amount_hkm].copy()

        return df

    def _tier1_enrich_fina(self, df: pd.DataFrame,
                           progress_callback=None) -> pd.DataFrame:
        """Phase B: Enrich pre-filtered stocks with hk_fina_indicator data."""
        if df.empty:
            return df

        cfg = self.config
        fina_ttl = cfg.cache_fina_indicator_ttl_hours * 3600
        records = []
        total = len(df)

        for i, (_, row) in enumerate(df.iterrows()):
            ts_code = row['ts_code']
            if progress_callback:
                progress_callback(i + 1, total, ts_code)

            # Check disk cache first
            cache_key = f'hk_fina_t1_{ts_code}'
            cached = self.cache.get(cache_key, fina_ttl)
            if cached is not None and not cached.empty:
                latest = cached.sort_values('end_date', ascending=False).iloc[0]
                rec = row.to_dict()
                for col in ['pe_ttm', 'pb_ttm', 'total_market_cap', 'dividend_rate',
                            'roe_avg', 'gross_profit_ratio', 'debt_asset_ratio',
                            'dps_hkd', 'divi_ratio', 'bps']:
                    rec[col] = latest.get(col)
                records.append(rec)
                continue

            # API call
            try:
                fina_df = self._safe_call(
                    'hk_fina_indicator', ts_code=ts_code,
                    fields=('ts_code,end_date,pe_ttm,pb_ttm,total_market_cap,'
                            'dividend_rate,roe_avg,gross_profit_ratio,debt_asset_ratio,'
                            'dps_hkd,divi_ratio,bps'))
            except Exception:
                continue

            if fina_df.empty:
                continue

            self.cache.put(cache_key, fina_df)

            latest = fina_df.sort_values('end_date', ascending=False).iloc[0]
            rec = row.to_dict()
            for col in ['pe_ttm', 'pb_ttm', 'total_market_cap', 'dividend_rate',
                        'roe_avg', 'gross_profit_ratio', 'debt_asset_ratio',
                        'dps_hkd', 'divi_ratio', 'bps']:
                rec[col] = latest.get(col)
            records.append(rec)

        if not records:
            return pd.DataFrame()
        return pd.DataFrame(records)

    # ---- Tier 1: Filter ----

    def _tier1_filter(self, df: pd.DataFrame) -> pd.DataFrame:
        """Phase C: Apply full Tier 1 filters on enriched data."""
        if df.empty:
            df = df.copy()
            df['channel'] = pd.Series(dtype='object')
            return df

        cfg = self.config

        # 1. Market cap filter (total_market_cap in 百万港元)
        df = df[df['total_market_cap'].notna()].copy()
        df = df[df['total_market_cap'] >= cfg.min_market_cap_hkm].copy()

        # 2. PB > 0 and <= max_pb
        df = df[df['pb_ttm'].notna()].copy()
        df = df[(df['pb_ttm'] > 0) & (df['pb_ttm'] <= cfg.max_pb)].copy()

        # 3. Dual-channel PE split
        pe_valid = df['pe_ttm'].notna()
        main_mask = pe_valid & (df['pe_ttm'] > 0) & (df['pe_ttm'] <= cfg.max_pe)
        obs_mask = pe_valid & (df['pe_ttm'] < 0)

        main_df = df[main_mask].copy()
        obs_df = df[obs_mask].copy()

        # 4. Dividend yield > 0 — main channel only
        main_df = main_df[main_df['dividend_rate'].notna() & (main_df['dividend_rate'] > 0)].copy()
        main_df['channel'] = 'main'

        # 5. Observation channel: top N by market cap
        obs_df = obs_df.sort_values('total_market_cap', ascending=False).head(cfg.obs_channel_limit)
        obs_df['channel'] = 'observation'

        result = pd.concat([main_df, obs_df], ignore_index=True)
        return result

    # ---- Tier 1: Rank & Cut ----

    def _tier1_rank_and_cut(self, df: pd.DataFrame) -> pd.DataFrame:
        if df.empty:
            return df

        cfg = self.config
        main = df[df['channel'] == 'main'].copy()
        obs = df[df['channel'] == 'observation'].copy()

        if not main.empty:
            dv_max = main['dividend_rate'].max()
            main['_dv_norm'] = main['dividend_rate'] / dv_max if dv_max > 0 else 0

            pe_inv = 1.0 / main['pe_ttm']
            pe_inv_max = pe_inv.max()
            main['_pe_norm'] = pe_inv / pe_inv_max if pe_inv_max > 0 else 0

            pb_inv = 1.0 / main['pb_ttm']
            pb_inv_max = pb_inv.max()
            main['_pb_norm'] = pb_inv / pb_inv_max if pb_inv_max > 0 else 0

            main['tier1_score'] = (
                cfg.dv_weight * main['_dv_norm'] +
                cfg.pe_weight * main['_pe_norm'] +
                cfg.pb_weight * main['_pb_norm']
            )
            main = main.sort_values('tier1_score', ascending=False)
            main = main.head(cfg.tier2_main_limit)
            main = main.drop(columns=['_dv_norm', '_pe_norm', '_pb_norm'])
        else:
            main['tier1_score'] = []

        obs['tier1_score'] = 0.0
        result = pd.concat([main, obs], ignore_index=True)
        return result

    # ---- Tier 2: Hard vetoes ----

    def _check_hard_vetoes(self, ts_code: str) -> tuple[bool, str]:
        """Check debt ratio as hard veto (no pledge/audit in HK)."""
        cfg = self.config

        # Check debt_asset_ratio from fina_indicator
        try:
            fi_df = self._cached_call('hk_fina_indicator', ts_code=ts_code)
            if not fi_df.empty:
                fi_df = fi_df.sort_values('end_date', ascending=False)
                debt_ratio = fi_df.iloc[0].get('debt_asset_ratio')
                if debt_ratio is not None and not (debt_ratio != debt_ratio):
                    if float(debt_ratio) > cfg.max_debt_ratio_veto:
                        return False, f'debt_ratio={debt_ratio:.1f}% > {cfg.max_debt_ratio_veto}%'
        except Exception:
            pass

        # Check negative equity from balance sheet
        try:
            bs_df = self._cached_call('hk_balancesheet', ts_code=ts_code)
            if not bs_df.empty:
                bs_pivoted = _pivot_hk_line_items(bs_df, HK_BALANCE_MAP)
                if not bs_pivoted.empty:
                    bs_pivoted = bs_pivoted.sort_values('end_date', ascending=False)
                    equity = bs_pivoted.iloc[0].get('total_hldr_eqy_exc_min_int')
                    if equity is not None and not (equity != equity) and float(equity) < 0:
                        return False, f'negative_equity={float(equity)/1e6:.0f}M'
        except Exception:
            pass

        return True, ''

    # ---- Tier 2: Financial quality ----

    def _check_financial_quality(self, ts_code: str, channel: str = 'main'
                                 ) -> tuple[bool, dict[str, Any]]:
        cfg = self.config
        metrics: dict[str, Any] = {}

        try:
            fi_df = self._cached_call('hk_fina_indicator', ts_code=ts_code)
        except Exception:
            return False, metrics

        if fi_df.empty:
            return False, metrics

        fi_df = fi_df.sort_values('end_date', ascending=False)
        annual = fi_df[fi_df['end_date'].str.endswith('1231')]
        row = annual.iloc[0] if not annual.empty else fi_df.iloc[0]

        roe = row.get('roe_avg')
        gm = row.get('gross_profit_ratio')
        debt = row.get('debt_asset_ratio')

        def _safe_float(v):
            if v is None:
                return None
            try:
                f = float(v)
                return None if f != f else f
            except (TypeError, ValueError):
                return None

        metrics['roe_waa'] = _safe_float(roe)
        metrics['gross_margin'] = _safe_float(gm)
        metrics['debt_to_assets'] = _safe_float(debt)

        # Observation channel: check if latest net profit is positive
        if channel == 'observation':
            try:
                inc_df = self._cached_call('hk_income', ts_code=ts_code)
                if not inc_df.empty:
                    inc_pivoted = _pivot_hk_line_items(inc_df, HK_INCOME_MAP)
                    if not inc_pivoted.empty:
                        inc_pivoted = inc_pivoted.sort_values('end_date', ascending=False)
                        np_val = inc_pivoted.iloc[0].get('n_income_attr_p')
                        np_f = _safe_float(np_val)
                        if np_f is None or np_f <= 0:
                            return False, metrics
            except Exception:
                return False, metrics

        if metrics['roe_waa'] is not None and metrics['roe_waa'] < cfg.min_roe:
            return False, metrics
        if metrics['gross_margin'] is not None and metrics['gross_margin'] < cfg.min_gross_margin:
            return False, metrics
        if metrics['debt_to_assets'] is not None and metrics['debt_to_assets'] > cfg.max_debt_ratio:
            return False, metrics

        return True, metrics

    # ---- Tier 2: Factor 2 penetration return ----

    def _extract_factor2_metrics(self, ts_code: str, total_market_cap_hkm: float
                                 ) -> dict[str, Any]:
        """Extract Factor 2 metrics: R = AA * M / market_cap."""
        result: dict[str, Any] = {}
        mkt_cap = total_market_cap_hkm  # already in 百万港元

        # Get risk-free rate (cached globally)
        if self._rf_cache is None:
            try:
                rf_df = self._cached_call('yc_cb', ts_code=None, curve_type='0')
                if not rf_df.empty:
                    rf_df = rf_df.sort_values('trade_date', ascending=False)
                    self._rf_cache = float(rf_df.iloc[0]['yield'])
            except Exception:
                pass

        rf = self._rf_cache
        result['Rf'] = rf
        result['II'] = max(3.5, rf + 2.0) if rf is not None else None

        # Get income and cashflow (line-item format)
        try:
            inc_df = self._cached_call('hk_income', ts_code=ts_code)
        except Exception:
            return result

        try:
            cf_df = self._cached_call('hk_cashflow', ts_code=ts_code)
        except Exception:
            cf_df = pd.DataFrame()

        if inc_df.empty:
            return result

        inc_pivoted = _pivot_hk_line_items(inc_df, HK_INCOME_MAP)
        cf_pivoted = _pivot_hk_line_items(cf_df, HK_CASHFLOW_MAP) if not cf_df.empty else pd.DataFrame()

        if inc_pivoted.empty:
            return result

        inc_pivoted = inc_pivoted.sort_values('end_date', ascending=False)

        # Compute 3yr avg payout ratio M from hk_fina_indicator (divi_ratio)
        payout_ratios = []
        try:
            fi_df = self._cached_call('hk_fina_indicator', ts_code=ts_code)
            if not fi_df.empty:
                fi_sorted = fi_df.sort_values('end_date', ascending=False)
                fi_annual = fi_sorted[fi_sorted['end_date'].str.endswith('1231')]
                for _, r in fi_annual.head(3).iterrows():
                    dr = r.get('divi_ratio')
                    if dr is not None and pd.notna(dr) and float(dr) > 0:
                        val = float(dr)
                        if val > 5:  # likely already in %
                            payout_ratios.append(val)
                        else:  # likely decimal
                            payout_ratios.append(val * 100)
        except Exception:
            pass

        if payout_ratios:
            M = sum(payout_ratios) / len(payout_ratios)
            result['M'] = M
        else:
            result['M'] = None

        # AA = OCF - CapEx (simplified for HK)
        AA = None
        if not cf_pivoted.empty:
            cf_sorted = cf_pivoted.sort_values('end_date', ascending=False)
            annual_cf = cf_sorted[cf_sorted['end_date'].str.endswith('1231')]
            if not annual_cf.empty:
                cf_row = annual_cf.iloc[0]

                def _sf(val):
                    if val is None:
                        return None
                    try:
                        f = float(val)
                        return None if f != f else f
                    except (TypeError, ValueError):
                        return None

                ocf = _sf(cf_row.get('n_cashflow_act'))
                capex = abs(_sf(cf_row.get('c_pay_acq_const_fiolta')) or 0)

                if ocf is not None:
                    AA = (ocf - capex) / 1e6  # HKD yuan -> millions

        result['AA'] = AA

        # R = AA * M / mkt_cap
        if AA is not None and result.get('M') is not None and mkt_cap > 0:
            R = AA * (result['M'] / 100) / mkt_cap * 100
            result['R'] = R
        else:
            result['R'] = None

        # Classify R vs II
        if result.get('R') is not None and result.get('II') is not None and result.get('Rf') is not None:
            R_val = result['R']
            II_val = result['II']
            Rf_val = result['Rf']
            if R_val < Rf_val:
                result['R_vs_II'] = 'below_rf'
            elif R_val < II_val * 0.5:
                result['R_vs_II'] = 'fail'
            elif R_val < II_val:
                result['R_vs_II'] = 'marginal'
            else:
                result['R_vs_II'] = 'pass'
        else:
            result['R_vs_II'] = None

        return result

    # ---- Tier 2: Factor 4 valuation metrics ----

    def _extract_factor4_metrics(self, ts_code: str, close: float,
                                 total_market_cap_hkm: float) -> dict[str, Any]:
        result: dict[str, Any] = {}
        mkt_cap = total_market_cap_hkm

        try:
            inc_df = self._cached_call('hk_income', ts_code=ts_code)
            bs_df = self._cached_call('hk_balancesheet', ts_code=ts_code)
            cf_df = self._cached_call('hk_cashflow', ts_code=ts_code)
        except Exception:
            return result

        def _sf(val):
            if val is None:
                return None
            try:
                f = float(val)
                return None if f != f else f
            except (TypeError, ValueError):
                return None

        inc_pivoted = _pivot_hk_line_items(inc_df, HK_INCOME_MAP) if not inc_df.empty else pd.DataFrame()
        bs_pivoted = _pivot_hk_line_items(bs_df, HK_BALANCE_MAP) if not bs_df.empty else pd.DataFrame()
        cf_pivoted = _pivot_hk_line_items(cf_df, HK_CASHFLOW_MAP) if not cf_df.empty else pd.DataFrame()

        # Process income
        oper_profit = None
        fin_exp = 0
        np_parent = None
        if not inc_pivoted.empty:
            inc_sorted = inc_pivoted.sort_values('end_date', ascending=False)
            annual_inc = inc_sorted[inc_sorted['end_date'].str.endswith('1231')]
            if not annual_inc.empty:
                latest = annual_inc.iloc[0]
                oper_profit = _sf(latest.get('operate_profit'))
                fin_exp = _sf(latest.get('finance_exp')) or 0
                np_parent = _sf(latest.get('n_income_attr_p'))
                if oper_profit is not None:
                    result['oper_profit'] = oper_profit / 1e6
                if np_parent is not None:
                    result['np_parent'] = np_parent / 1e6

        # Process balance sheet
        cash = 0
        ibd = 0.0
        ta = 0
        if not bs_pivoted.empty:
            bs_sorted = bs_pivoted.sort_values('end_date', ascending=False)
            annual_bs = bs_sorted[bs_sorted['end_date'].str.endswith('1231')]
            if not annual_bs.empty:
                latest = annual_bs.iloc[0]
                cash = (_sf(latest.get('money_cap')) or 0) / 1e6
                ta = (_sf(latest.get('total_assets')) or 0) / 1e6

                for c in ['st_borr', 'lt_borr', 'bond_payable']:
                    v = _sf(latest.get(c))
                    if v:
                        ibd += v / 1e6

                result['cash'] = cash
                result['ibd'] = ibd
                result['total_assets'] = ta

                if ta > 0:
                    intang = (_sf(latest.get('intang_assets')) or 0) / 1e6
                    result['goodwill_ratio'] = intang / ta * 100

        # Process cashflow
        fcf_list = []
        da = 0
        if not cf_pivoted.empty:
            cf_sorted = cf_pivoted.sort_values('end_date', ascending=False)
            annual_cf = cf_sorted[cf_sorted['end_date'].str.endswith('1231')]
            if not annual_cf.empty:
                latest = annual_cf.iloc[0]
                ocf = (_sf(latest.get('n_cashflow_act')) or 0) / 1e6
                capex = abs((_sf(latest.get('c_pay_acq_const_fiolta')) or 0)) / 1e6
                da = (_sf(latest.get('depr_fa_coga_dpba')) or 0) / 1e6

                result['da'] = da
                result['fcf'] = ocf - capex
                if mkt_cap > 0:
                    result['fcf_yield'] = (ocf - capex) / mkt_cap * 100

            # FCF consistency (5 years)
            for _, row in annual_cf.head(5).iterrows():
                o = _sf(row.get('n_cashflow_act'))
                c = _sf(row.get('c_pay_acq_const_fiolta'))
                if o is not None and c is not None:
                    fcf_list.append(o - abs(c))

        if fcf_list:
            result['fcf_positive_years'] = sum(1 for f in fcf_list if f > 0)
            result['fcf_total_years'] = len(fcf_list)
            result['fcf_consistency'] = result['fcf_positive_years'] / result['fcf_total_years']

        # Composite metrics
        net_debt_m = ibd - cash
        ebitda = None
        if oper_profit is not None:
            ebitda = oper_profit / 1e6 + fin_exp / 1e6 + da

        if ebitda is not None:
            result['ebitda'] = ebitda
            ev = mkt_cap + net_debt_m
            result['ev'] = ev
            if ebitda > 0:
                result['ev_ebitda'] = ev / ebitda
                result['net_debt_ebitda'] = net_debt_m / ebitda

        net_cash = -net_debt_m
        if np_parent is not None:
            np_m = np_parent / 1e6
            if np_m > 0:
                result['cash_adj_pe'] = (mkt_cap - net_cash) / np_m

        return result

    # ---- Tier 2: Floor price ----

    def _extract_floor_price(self, ts_code: str, close: float,
                             total_market_cap_hkm: float) -> dict[str, Any]:
        result: dict[str, Any] = {}
        total_shares = total_market_cap_hkm * 1e6 / close if close > 0 else 0

        try:
            bs_df = self._cached_call('hk_balancesheet', ts_code=ts_code)
            cf_df = self._cached_call('hk_cashflow', ts_code=ts_code)
        except Exception:
            return result

        def _sf(val):
            if val is None:
                return None
            try:
                f = float(val)
                return None if f != f else f
            except (TypeError, ValueError):
                return None

        bs_pivoted = _pivot_hk_line_items(bs_df, HK_BALANCE_MAP) if not bs_df.empty else pd.DataFrame()
        cf_pivoted = _pivot_hk_line_items(cf_df, HK_CASHFLOW_MAP) if not cf_df.empty else pd.DataFrame()

        baselines = []

        # 1. Net liquid assets / share
        if not bs_pivoted.empty:
            bs_sorted = bs_pivoted.sort_values('end_date', ascending=False)
            annual_bs = bs_sorted[bs_sorted['end_date'].str.endswith('1231')]
            if not annual_bs.empty and total_shares > 0:
                latest = annual_bs.iloc[0]
                cash_val = _sf(latest.get('money_cap')) or 0
                ibd_val = 0.0
                for c in ['st_borr', 'lt_borr', 'bond_payable']:
                    v = _sf(latest.get(c))
                    if v:
                        ibd_val += v
                nla = (cash_val - ibd_val) / total_shares
                baselines.append(('net_liquid_assets', nla))

                # 2. BVPS
                equity = _sf(latest.get('total_hldr_eqy_exc_min_int')) or 0
                bvps = equity / total_shares
                baselines.append(('bvps', bvps))

        # 3. 10-year low via yfinance
        try:
            import yfinance as yf
            parts = ts_code.split('.')
            if len(parts) == 2 and parts[1] == 'HK':
                yf_ticker = str(int(parts[0])) + '.HK'
            else:
                yf_ticker = ts_code
            hist = yf.Ticker(yf_ticker).history(period='10y', interval='1wk')
            if not hist.empty:
                min_close = hist['Close'].dropna().min()
                if min_close is not None:
                    baselines.append(('10yr_low', float(min_close)))
        except Exception:
            pass

        # 4. Dividend implied price
        rf = self._rf_cache
        try:
            fi_df = self._cached_call('hk_fina_indicator', ts_code=ts_code)
            if not fi_df.empty and rf is not None:
                fi_sorted = fi_df.sort_values('end_date', ascending=False)
                recent_dps = []
                for _, row in fi_sorted.head(3).iterrows():
                    v = _sf(row.get('dps_hkd'))
                    if v is not None and v > 0:
                        recent_dps.append(v)
                if recent_dps:
                    avg_dps = sum(recent_dps) / len(recent_dps)
                    discount = max(rf / 100, 0.03)
                    implied = avg_dps / discount
                    baselines.append(('dividend_implied', implied))
        except Exception:
            pass

        # 5. Pessimistic FCF capitalization
        if not cf_pivoted.empty and rf is not None and rf > 0 and total_shares > 0:
            cf_sorted = cf_pivoted.sort_values('end_date', ascending=False)
            annual_cf = cf_sorted[cf_sorted['end_date'].str.endswith('1231')]
            fcf_list = []
            for _, row in annual_cf.head(5).iterrows():
                o = _sf(row.get('n_cashflow_act'))
                c = _sf(row.get('c_pay_acq_const_fiolta'))
                if o is not None and c is not None:
                    fcf_list.append(o - abs(c))
            if fcf_list and min(fcf_list) > 0:
                min_fcf = min(fcf_list)
                cap_price = min_fcf / (rf / 100) / total_shares
                baselines.append(('pessimistic_fcf', cap_price))

        result['baselines'] = baselines

        valid_prices = [v for _, v in baselines]
        if valid_prices:
            composite = sum(valid_prices) / len(valid_prices)
            result['composite_baseline'] = composite
            result['premium'] = (close / composite - 1) * 100 if composite > 0 else None
        else:
            result['composite_baseline'] = None
            result['premium'] = None

        return result

    # ---- Tier 2: Analyze single stock ----

    def _analyze_single_stock(self, row: pd.Series) -> dict[str, Any] | None:
        ts_code = row['ts_code']
        channel = row.get('channel', 'main')
        close = float(row.get('close', 0))
        total_mkt_cap = float(row.get('total_market_cap', 0))

        stock_result = {
            'ts_code': ts_code,
            'name': row.get('name', ''),
            'channel': channel,
            'close': close,
            'total_market_cap': total_mkt_cap,
            'pe_ttm': row.get('pe_ttm'),
            'pb': row.get('pb_ttm'),
            'dv_ttm': row.get('dividend_rate'),
        }

        # Step 1: Hard vetoes
        passed, reason = self._check_hard_vetoes(ts_code)
        if not passed:
            stock_result['veto'] = reason
            self._clear_stock_cache(ts_code)
            return None

        # Step 2: Financial quality
        fq_passed, fq_metrics = self._check_financial_quality(ts_code, channel)
        stock_result.update(fq_metrics)
        if not fq_passed:
            stock_result['veto'] = 'financial_quality'
            self._clear_stock_cache(ts_code)
            return None

        # Step 3: Factor 2 (penetration return)
        f2 = self._extract_factor2_metrics(ts_code, total_mkt_cap)
        stock_result['M'] = f2.get('M')
        stock_result['R'] = f2.get('R')
        stock_result['II'] = f2.get('II')
        stock_result['Rf'] = f2.get('Rf')
        stock_result['R_vs_II'] = f2.get('R_vs_II')

        # Step 4: Factor 4 (valuation)
        f4 = self._extract_factor4_metrics(ts_code, close, total_mkt_cap)
        stock_result['ev_ebitda'] = f4.get('ev_ebitda')
        stock_result['cash_adj_pe'] = f4.get('cash_adj_pe')
        stock_result['fcf_yield'] = f4.get('fcf_yield')
        stock_result['net_debt_ebitda'] = f4.get('net_debt_ebitda')
        stock_result['goodwill_ratio'] = f4.get('goodwill_ratio')
        stock_result['fcf_consistency'] = f4.get('fcf_consistency')

        # Step 5: Floor price
        fp = self._extract_floor_price(ts_code, close, total_mkt_cap)
        stock_result['floor_baseline'] = fp.get('composite_baseline')
        stock_result['floor_premium'] = fp.get('premium')

        self._clear_stock_cache(ts_code)
        return stock_result

    # ---- Scoring ----

    def _compute_rankings(self, df: pd.DataFrame) -> pd.DataFrame:
        if df.empty:
            return df
        cfg = self.config
        df = df.copy()

        for col in ['roe_waa', 'fcf_yield', 'R']:
            if col in df.columns:
                df[f'{col}_pctile'] = df[col].rank(pct=True, na_option='bottom')
            else:
                df[f'{col}_pctile'] = 0.0

        for col in ['ev_ebitda', 'floor_premium']:
            if col in df.columns:
                df[f'{col}_pctile'] = 1.0 - df[col].rank(pct=True, na_option='top')
            else:
                df[f'{col}_pctile'] = 0.0

        df['composite_score'] = (
            cfg.weight_roe * df['roe_waa_pctile'] +
            cfg.weight_fcf_yield * df['fcf_yield_pctile'] +
            cfg.weight_penetration_r * df['R_pctile'] +
            cfg.weight_ev_ebitda * df['ev_ebitda_pctile'] +
            cfg.weight_floor_premium * df['floor_premium_pctile']
        )

        df = df.sort_values('composite_score', ascending=False)
        return df

    # ---- Pipeline ----

    def run(self, tier2_limit: int | None = None,
            progress_callback=None) -> pd.DataFrame:
        """Run Tier 2 analysis on already-ranked Tier 1 results."""
        ranked = self._ranked_df
        if ranked is None or ranked.empty:
            print('No ranked data. Run Tier 1 first.')
            return pd.DataFrame()

        if tier2_limit is not None:
            ranked = ranked.head(tier2_limit)

        total = len(ranked)
        print(f'\n=== Tier 2: 深度分析 ({total} 只) ===')

        results = []
        for i, (_, row) in enumerate(ranked.iterrows()):
            ts_code = row['ts_code']
            if progress_callback:
                progress_callback(i + 1, total, ts_code)
            else:
                name = row.get('name', '')
                print(f'  [{i+1}/{total}] {ts_code} {name}...', end='')

            stock_result = self._analyze_single_stock(row)
            if stock_result is None:
                if not progress_callback:
                    print(' VETOED')
            else:
                results.append(stock_result)
                if not progress_callback:
                    print(' OK')

        if not results:
            return pd.DataFrame()

        result_df = pd.DataFrame(results)
        result_df = self._compute_rankings(result_df)
        print(f'\n=== 结果: {len(result_df)} 只通过 ===')
        return result_df

    # ---- Export ----

    def export_csv(self, df: pd.DataFrame, path: str) -> None:
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f'Exported to {path}')

    def export_html(self, df: pd.DataFrame, path: str) -> None:
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        display_cols = [c for c in [
            'ts_code', 'name', 'close', 'pe_ttm', 'pb', 'dv_ttm',
            'roe_waa', 'gross_margin', 'fcf_yield', 'R', 'ev_ebitda',
            'floor_premium', 'composite_score'
        ] if c in df.columns]
        display_df = df[display_cols].copy()
        table_html = display_df.to_html(index=False, float_format='%.2f',
                                        classes='screener-table')
        ts_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        count = len(df)
        parts = [
            '<!DOCTYPE html><html><head><meta charset="utf-8">',
            '<title>龟龟港股选股器 Results</title>',
            '<style>',
            'body { font-family: -apple-system, sans-serif; margin: 20px; }',
            '.screener-table { border-collapse: collapse; width: 100%; }',
            '.screener-table th { background: #2c3e50; color: white; padding: 8px 12px; text-align: left; }',
            '.screener-table td { border-bottom: 1px solid #ddd; padding: 6px 12px; }',
            '.screener-table tr:hover { background: #f5f5f5; }',
            '</style></head><body>',
            '<h1>龟龟港股选股器 — 筛选结果</h1>',
            f'<p>生成时间: {ts_str} | 共 {count} 只股票</p>',
            table_html,
            '</body></html>',
        ]
        with open(path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(parts))
        print(f'Exported to {path}')


print('Engine loaded: HKScreenerConfig, ScreenerCache, HKScreener')


In [ ]:
# Cell 2: 配置参数与初始化
import matplotlib
import matplotlib.pyplot as plt

# 中文字体设置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# ===== 配置参数（可自定义修改）=====
# ============================================================

config = HKScreenerConfig(
    # --- Tier 1: 硬性过滤 ---
    min_listing_years=3,           # 最低上市年限
    min_market_cap_hkm=5000.0,     # 最低市值（百万港元，即50亿港元）
    min_daily_amount_hkm=5.0,      # 最低日成交额（百万港元）
    max_pb=8.0,                    # PB 上限
    max_pe=30.0,                   # PE 上限
    obs_channel_limit=30,          # 观察通道（PE<0）最多保留数量

    # --- Tier 1: 排序权重（三者之和建议为1.0）---
    dv_weight=0.4,                 # 股息率权重
    pe_weight=0.3,                 # 1/PE 权重
    pb_weight=0.3,                 # 1/PB 权重
    tier2_main_limit=100,          # 主通道入 Tier 2 最大数量

    # --- Tier 2: 硬性否决 ---
    max_debt_ratio_veto=85.0,      # 负债率否决线(%)，超过一票否决

    # --- Tier 2: 质量门槛 ---
    min_roe=8.0,                   # ROE 最低门槛（%）
    min_gross_margin=15.0,         # 毛利率下限（%）
    max_debt_ratio=70.0,           # 资产负债率上限（%）

    # --- Tier 2: 综合评分权重（五者之和必须=1.0）---
    weight_penetration_r=0.25,     # 穿透回报率 R（策略核心）
    weight_roe=0.20,               # ROE
    weight_fcf_yield=0.20,         # FCF 收益率
    weight_floor_premium=0.20,     # 底价溢价率
    weight_ev_ebitda=0.15,         # EV/EBITDA
)

# 参数校验
errors = config.validate()
if errors:
    for e in errors:
        print(f'Warning: {e}')
else:
    print('Config OK')

screener = HKScreener(config=config)

# 显示配置
cfg_df = pd.DataFrame([
    {'参数': k, '值': v}
    for k, v in config.to_dict().items()
    if not k.startswith('cache')
])
display(cfg_df.style.hide(axis='index').set_caption('港股筛选器配置'))

# 显示缓存状态
cache_dir = config.cache_dir
if os.path.isdir(cache_dir):
    files = [f for f in os.listdir(cache_dir) if f.endswith('.parquet')]
    total_size = sum(os.path.getsize(os.path.join(cache_dir, f))
                     for f in os.listdir(cache_dir) if os.path.isfile(os.path.join(cache_dir, f)))
    size_mb = total_size / (1024 * 1024)
    print(f'Cache: {cache_dir}')
    print(f'  Parquet files: {len(files)}')
    print(f'  Size: {size_mb:.2f} MB')
else:
    print(f'Cache: {cache_dir} (not yet created)')

## Step 2: Tier 1 批量粗筛

港股 Tier 1 分三个阶段：

**Phase A** — `hk_basic` + `hk_daily` 批量获取，预筛（上市年限、成交额、停牌股）

**Phase B** — 对预筛通过的股票逐只拉取 `hk_fina_indicator`（PE/PB/市值/股息率等），缓存7天

**Phase C** — 施加完整过滤条件：
1. **市值 >= 50亿港元** — 排除微盘股
2. **PB > 0 且 <= 8** — 排除破净和泡沫
3. **双通道 PE**：主通道 (0 < PE <= 30, 股息率 > 0) + 观察通道 (PE < 0, 按市值取前30)

> Phase B 首次运行需要 3-8 分钟（取决于预筛数量），后续运行利用缓存可在几秒内完成。

In [ ]:
# Cell 3: Tier 1 批量粗筛 (3-phase)

# Phase A: hk_basic + hk_daily 批量获取 + 预筛
print('=== Phase A: 批量获取 hk_basic + hk_daily ===')
bulk_df = screener._tier1_bulk_data()
print(f'全港股数量: {len(bulk_df)}')

pre_filtered = screener._tier1_pre_filter(bulk_df)
print(f'预筛后: {len(pre_filtered)} 只 (上市>=3年, 成交额>=5M HKD, 非停牌)')

# Phase B: 逐只拉取 hk_fina_indicator (首次运行较慢，缓存后秒级)
print(f'\n=== Phase B: 拉取 hk_fina_indicator ({len(pre_filtered)} 只) ===')
try:
    from tqdm.notebook import tqdm
    pbar = tqdm(total=len(pre_filtered), desc='Phase B')
    def _t1_progress(current, total, ts_code):
        pbar.n = current
        pbar.total = total
        pbar.set_postfix(stock=ts_code)
        pbar.refresh()
    enriched = screener._tier1_enrich_fina(pre_filtered, progress_callback=_t1_progress)
    pbar.close()
except ImportError:
    enriched = screener._tier1_enrich_fina(pre_filtered)
print(f'获取到财务指标: {len(enriched)} 只')

# Phase C: 完整过滤
print(f'\n=== Phase C: 完整过滤 ===')
filtered = screener._tier1_filter(enriched)

if filtered.empty:
    main = obs = filtered
    print('Tier 1 筛选结果: 无股票通过筛选')
else:
    main = filtered[filtered['channel'] == 'main']
    obs = filtered[filtered['channel'] == 'observation']
    print(f'Tier 1 筛选结果:')
    print(f'  通过筛选: {len(filtered)} 只')
    print(f'  主通道: {len(main)} 只 (PE > 0 且 <= {config.max_pe}, 股息率 > 0)')
    print(f'  观察通道: {len(obs)} 只 (PE < 0, 亏损股, 按市值取前{config.obs_channel_limit})')

# 市值分布图
if not filtered.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    (filtered['total_market_cap'] / 1000).plot.hist(ax=axes[0], bins=30, edgecolor='black')
    axes[0].set_title('市值分布 (十亿港元)')
    axes[0].set_xlabel('市值 (十亿港元)')
    axes[0].set_ylabel('数量')

    if not main.empty:
        main['pe_ttm'].plot.hist(ax=axes[1], bins=30, edgecolor='black', color='steelblue')
        axes[1].set_title('PE(TTM) 分布 (主通道)')
        axes[1].set_xlabel('PE (TTM)')
        axes[1].set_ylabel('数量')

    plt.tight_layout()
    plt.show()

## Step 3: Tier 1 排序与截断

通过粗筛的股票按**综合得分**排序，取 Top 130 进入 Tier 2。

### 排序公式

```
Tier1 Score = 0.4 × dv_norm + 0.3 × (1/PE)_norm + 0.3 × (1/PB)_norm
```

- **`dv_norm`**：股息率归一化（min-max scaling 到 0~1）
- **`(1/PE)_norm`**：PE 倒数归一化（PE 越低越好）
- **`(1/PB)_norm`**：PB 倒数归一化（PB 越低越好）

### 双通道

- **主通道**（PE > 0 且有股息）：取 Top 100
- **观察通道**（PE < 0，亏损）：按市值 Top 30，不参与排分

In [ ]:
# Cell 4: Tier 1 排序与截断
ranked = screener._tier1_rank_and_cut(filtered)
main_ranked = ranked[ranked['channel'] == 'main']
obs_ranked = ranked[ranked['channel'] == 'observation']
print(f'进入 Tier 2: {len(ranked)} 只 (主通道 {len(main_ranked)}, 观察通道 {len(obs_ranked)})')

# Store for Tier 2 use
screener._ranked_df = ranked

# 展示主通道候选列表
display_cols = ['ts_code', 'name', 'close', 'pe_ttm', 'pb_ttm',
                'dividend_rate', 'total_market_cap', 'tier1_score', 'channel']
display_cols = [c for c in display_cols if c in ranked.columns]
display(main_ranked[display_cols].head(20).style
        .format({'pe_ttm': '{:.1f}', 'pb_ttm': '{:.2f}', 'dividend_rate': '{:.2f}',
                 'tier1_score': '{:.3f}', 'total_market_cap': '{:,.0f}'},
                na_rep='—')
        .set_caption(f'主通道 Top 20 / {len(main_ranked)}'))

# 展示观察通道
if not obs_ranked.empty:
    display(obs_ranked[display_cols].head(20).style
            .format({'pe_ttm': '{:.1f}', 'pb_ttm': '{:.2f}', 'dividend_rate': '{:.2f}',
                     'tier1_score': '{:.3f}', 'total_market_cap': '{:,.0f}'},
                    na_rep='—')
            .set_caption(f'观察通道 Top 20 / {len(obs_ranked)}'))

# PE/PB/股息率 散点图
if not ranked.empty:
    main_r = ranked[ranked['channel'] == 'main']
    if not main_r.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        scatter = ax.scatter(main_r['pe_ttm'], main_r['pb_ttm'],
                           c=main_r['dividend_rate'], cmap='RdYlGn',
                           s=40, alpha=0.6)
        plt.colorbar(scatter, label='股息率 (%)')
        ax.set_xlabel('PE (TTM)')
        ax.set_ylabel('PB')
        ax.set_title('Tier 1 候选: PE vs PB (颜色=股息率)')
        plt.tight_layout()
        plt.show()

## Step 4: Tier 2 逐股深度分析

对 Tier 1 候选的每只股票，逐一拉取完整财务数据并进行多维度评估。

**第一关：硬性否决** — 负债率 > 85% 或 负权益 → 直接拒绝

**第二关：质量门槛** — ROE >= 8%, 毛利率 >= 15%, 负债率 <= 70%

**第三关：Factor 2** — 穿透回报率 R = AA × M / 市值

**第四关：Factor 4** — EV/EBITDA, FCF yield, 现金调整PE

**第五关：底价** — 5种方法取平均

> 港股与A股差异：无质押/审计否决，改用更严格的负债率门槛 (85%)。

In [ ]:
# Cell 5: Tier 2 逐股深度分析
# 可调整 TIER2_LIMIT 控制分析数量 (None = 全部, 5 = 快速测试)
TIER2_LIMIT = 50  # 修改此值: None=全部130只, 5=快速测试

try:
    from tqdm.notebook import tqdm
    pbar = tqdm(total=min(TIER2_LIMIT or len(ranked), len(ranked)),
                desc='Tier 2 分析')
    def _progress(current, total, ts_code):
        pbar.n = current
        pbar.total = total
        pbar.set_postfix(stock=ts_code)
        pbar.refresh()

    result = screener.run(tier2_limit=TIER2_LIMIT, progress_callback=_progress)
    pbar.close()
except ImportError:
    result = screener.run(tier2_limit=TIER2_LIMIT)

print(f'\nTier 2 通过: {len(result)} 只')

## Step 5: 综合评分与排名

Tier 2 通过的股票按 **5个维度** 加权计算综合得分：

```
Score = 0.20×ROE + 0.20×FCF + 0.25×R + 0.15×EV/EBITDA + 0.20×Floor
```

| 维度 | 权重 | 方向 |
|------|------|------|
| ROE | 20% | 越高越好 |
| FCF yield | 20% | 越高越好 |
| Penetration R | 25% | 越高越好 |
| EV/EBITDA | 15% | 越低越好 |
| Floor premium | 20% | 越低越好 |

In [ ]:
# Cell 6: 综合评分与排名
if not result.empty:
    display_cols = ['ts_code', 'name', 'close', 'pe_ttm', 'pb',
                    'dv_ttm', 'roe_waa', 'gross_margin', 'fcf_yield', 'R',
                    'ev_ebitda', 'floor_premium', 'composite_score']
    display_cols = [c for c in display_cols if c in result.columns]

    top50 = result[display_cols].head(50)

    styled = (top50.style
        .format({
            'pe_ttm': '{:.1f}', 'pb': '{:.2f}', 'dv_ttm': '{:.2f}',
            'roe_waa': '{:.1f}%', 'gross_margin': '{:.1f}%',
            'fcf_yield': '{:.2f}%', 'R': '{:.2f}%',
            'ev_ebitda': '{:.1f}x', 'floor_premium': '{:.1f}%',
            'composite_score': '{:.3f}'
        }, na_rep='—')
        .background_gradient(subset=['composite_score'], cmap='RdYlGn')
        .set_caption(f'综合排名 Top {len(top50)}'))
    display(styled)

    # 综合得分分布
    fig, ax = plt.subplots(figsize=(10, 5))
    result['composite_score'].plot.hist(ax=ax, bins=20, edgecolor='black')
    ax.set_title('综合得分分布')
    ax.set_xlabel('Composite Score')
    plt.tight_layout()
    plt.show()
else:
    print('无结果')

## Step 6: 结果导出

将筛选结果导出为 CSV / HTML / Excel 三种格式到 `output/` 目录。

In [ ]:
# Cell 7: 结果导出
if not result.empty:
    os.makedirs('../output', exist_ok=True)

    csv_path = '../output/screener_hk_results.csv'
    screener.export_csv(result, csv_path)

    html_path = '../output/screener_hk_results.html'
    screener.export_html(result, html_path)

    try:
        xlsx_path = '../output/screener_hk_results.xlsx'
        result.to_excel(xlsx_path, index=False)
        print(f'Excel: {xlsx_path}')
    except Exception as e:
        print(f'Excel export skipped: {e}')

    print(f'\n共导出 {len(result)} 只港股')
else:
    print('无数据可导出')

## Step 7: 单股深入查看

输入任意港股代码，查看该股详细指标及在全样本中的**百分位位置**。

> 修改 `STOCK_CODE` 变量即可切换查看不同股票。

In [ ]:
# Cell 8 (可选): 单股深入查看
STOCK_CODE = '00700.HK'  # 修改此处查看不同股票 (e.g. 00700.HK 腾讯)

if not result.empty and STOCK_CODE in result['ts_code'].values:
    stock = result[result['ts_code'] == STOCK_CODE].iloc[0]

    print(f'=== {stock["name"]} ({STOCK_CODE}) ===')
    print(f'当前价格: {stock.get("close", "—")} HKD')
    score = stock.get('composite_score')
    if score is not None:
        print(f'综合得分: {score:.3f}')
    rank = (result['ts_code'] == STOCK_CODE).values.argmax() + 1
    print(f'排名: {rank}/{len(result)}')
    print()

    metrics = [
        ('ROE(平均)', 'roe_waa', '%'),
        ('毛利率', 'gross_margin', '%'),
        ('FCF收益率', 'fcf_yield', '%'),
        ('穿透回报率R', 'R', '%'),
        ('EV/EBITDA', 'ev_ebitda', 'x'),
        ('底价溢价率', 'floor_premium', '%'),
        ('无形资产/总资产', 'goodwill_ratio', '%'),
        ('净负债/EBITDA', 'net_debt_ebitda', 'x'),
    ]

    for label, col, unit in metrics:
        val = stock.get(col)
        if val is not None and val == val:
            pctile = (result[col] <= val).mean() * 100 if col in result.columns else None
            pctile_str = f' (百分位: {pctile:.0f}%)' if pctile is not None else ''
            print(f'  {label}: {val:.2f}{unit}{pctile_str}')
        else:
            print(f'  {label}: —')
else:
    if result.empty:
        print('请先运行 Cell 5-6')
    else:
        print(f'{STOCK_CODE} 不在结果中。可用股票:')
        print(result['ts_code'].head(10).tolist())